# Proyecto Final: Introducción a Data Science y Machine Learning
**Facultad de Ingeniería - Universidad de San Carlos de Guatemala** 

**Autor:** David Guerra, Ingeniero Industrial

---

## 1. Comprensión del problema de negocio

### Descripción del contexto del problema
En el área de laminación de una planta de empaques flexibles, el departamento de planificación utiliza tablas estáticas basadas únicamente en el tamaño de la orden de producción para calcular las pérdidas de material esperadas. Estas tablas aplican un porcentaje global que no distingue entre la "merma" (material perdido típicamente en los arranques o seteos del equipo) y el "desperdicio" (pérdidas generadas durante la operación normal), ni toman en cuenta las variables específicas del proceso de laminación. Esto genera imprecisiones constantes al momento de asignar los materiales necesarios a la máquina.

### Explicación del objetivo del análisis
El objetivo es identificar y modelar el comportamiento de las pérdidas en la laminadora, encontrando la relación entre las características de las órdenes y los kilogramos reales de pérdida reportados por los operadores. Esto permitirá actualizar la metodología de planificación, pasando de un porcentaje estático a un cálculo dinámico y fundamentado en datos.

### Formulación del problema
Predecir de forma independiente la cantidad en kilogramos de merma y de desperdicio que se generarán en una orden de producción de laminación, basándose en las características técnicas y operativas de dicha orden.

### Identificación de Variables Objetivo
* **Kilogramos de merma**
* **Kilogramos de desperdicio**

### Tipo de problema
Corresponde a un **problema de regresión**, ya que el objetivo es predecir valores numéricos continuos (kilogramos). Al tener dos salidas deseadas, el enfoque será desarrollar modelos de regresión independientes para cada tipo de pérdida.

### Descripción de la utilidad práctica del modelo
La implementación de este modelo mejorará significativamente la precisión en la planificación de la producción. Al predecir los kilogramos exactos de merma y desperdicio, se asignará la cantidad precisa de material base, garantizando la entrega del 100% de la orden al cliente, optimizando los costos de materia prima y reduciendo las variaciones en el piso de producción. Adicionalmente, este proyecto sentará las bases para identificar comportamientos específicos en el proceso (como los factores que disparan la merma durante el arranque de las máquinas), abriendo la puerta a futuras iniciativas de mejora continua y reducción sistemática de pérdidas.

---

## 2. Recolección de Datos

### Fuentes de datos
Los datos fueron extraídos de los registros históricos de producción del área de laminación de la planta de empaques flexibles. Se cuenta con un conjunto de datos en formato CSV que documenta las características de las órdenes de producción y los kilogramos reales de pérdida reportados por los operadores.

### Descripción del conjunto de datos
El *dataset* está compuesto por registros históricos donde cada fila representa una orden de producción procesada en una máquina laminadora. Las variables a utilizar son:

* **Fecha:** Fecha en la que se procesó la orden (útil para análisis temporal, aunque podría descartarse en el modelo final si no hay estacionalidad).
* **Turno:** Variable categórica que indica el turno de trabajo (ej. A, B).
* **Categoría:** Variable categórica que describe los materiales o sustratos laminados (ej. BOPP/COEX, BOPP/Metalizado).
* **Máquina:** Variable categórica que identifica el equipo utilizado (ej. SUPER SIMPLEX, SIMPLEX 2).
* **Gramaje:** Variable numérica continua que representa el peso por metro cuadrado del material.
* **Ancho:** Variable numérica continua que indica el ancho de la bobina procesada.
* **Peso_kg:** Variable numérica continua que representa el tamaño de la orden en kilogramos.
* **Metros_lineales:** Variable numérica continua que indica la longitud total de la orden.
---

## 3. Análisis Exploratorio de Datos (EDA)
*(Nota: Las gráficas y el código detallado de esta sección se encuentran en el apartado de ejecución del cuaderno).*
Durante esta fase se analizó la distribución de las variables y se buscaron correlaciones iniciales. Se identificó que variables como el `peso_kg` y los `metros_lineales` tienen una fuerte relación positiva con la cantidad de merma generada. Asimismo, se observó que la variable objetivo (`merma_kg`) presenta una dispersión considerable, indicando la presencia de valores atípicos (*outliers*) que corresponden a órdenes donde el proceso de arranque fue inusualmente problemático.

---

## 4. Ingeniería de Características (Feature Engineering)
Para que los algoritmos de Machine Learning pudieran procesar la información correctamente y evitar el sobreajuste (*data leakage*), se aplicaron las siguientes transformaciones matemáticas, calculadas estrictamente sobre el conjunto de entrenamiento:

1. **Manejo de Fechas:** Los algoritmos no procesan formatos de fecha estándar. Se extrajo el "día de la semana" como una nueva variable numérica (para evaluar si los arranques de semana generan más merma) y se descartó la fecha original.
2. **Imputación de Valores Nulos:** Se detectaron valores faltantes en las columnas `gramaje` y `ancho` en el conjunto de prueba. Estos fueron rellenados utilizando la mediana del conjunto de entrenamiento.
3. **Codificación Categórica (One-Hot Encoding):** Las variables de texto (`turno`, `categoria`, `maquina`) se transformaron en variables binarias (0 y 1). Se utilizó el parámetro `drop_first=True` para evitar la multicolinealidad perfecta.
4. **Escalamiento de Variables:** Dado que variables como `peso_kg` manejan rangos en los miles y `gramaje` en decenas, se aplicó un `StandardScaler` para estandarizar las magnitudes numéricas y evitar que el modelo sesgara su aprendizaje hacia los números más grandes.

---

## 5. Modelado y Optimización
Para este problema de regresión, se seleccionó el algoritmo **Random Forest Regressor** (Bosque Aleatorio). Este modelo es ideal para entornos industriales porque maneja de forma excelente combinaciones de variables numéricas y categóricas sin asumir relaciones estrictamente lineales.

### Control de Sobreajuste (Hyperparameter Tuning)
En las primeras iteraciones, el modelo base presentó un fuerte sobreajuste (*overfitting*), memorizando el set de entrenamiento ($R^2$ de 0.88) pero fallando en la generalización con datos nuevos. Para corregirlo, se implementó una optimización de hiperparámetros mediante **GridSearchCV** con validación cruzada (CV=5). Se limitó la profundidad máxima de los árboles (`max_depth`) y se aumentó el número mínimo de muestras por hoja (`min_samples_leaf`), obligando al modelo a aprender patrones generales en lugar de casos aislados.

---

## 6. Evaluación del Modelo
Tras aplicar la optimización, se obtuvieron las siguientes métricas comparativas entre el conjunto de Entrenamiento (213 órdenes) y el de Prueba (54 órdenes):

* **Error Absoluto Medio (MAE):** Entrenamiento: 6.65 kg | Prueba: 11.03 kg
* **Coeficiente de Determinación ($R^2$):** Entrenamiento: 0.6378 | Prueba: 0.2090

**Interpretación de Resultados:**
La optimización logró su objetivo principal: el modelo dejó de memorizar los datos (el $R^2$ de entrenamiento bajó a un nivel realista de 0.63). Sin embargo, el rendimiento en el conjunto de prueba sigue siendo bajo (0.20). Esto diagnostica claramente una limitación de **volumen de datos**. Con 15 variables predictoras generadas tras el *One-Hot Encoding*, 213 registros son insuficientes para que el algoritmo consolide reglas matemáticas robustas para cada combinación posible de máquina, material y tamaño de orden.

---

## 7. Propuesta de Implementación
Para trasladar el valor de este análisis al departamento de Planificación, no se recomienda volver a la creación de rangos estáticos agrupados por tamaño. En su lugar, se propone el desarrollo de una **"Calculadora de Merma Dinámica"**.
Esta herramienta sería una interfaz sencilla (o un *script* integrado al ERP) donde el planificador ingresa las variables de la orden (Máquina, Material, Peso, Ancho) y el modelo de Machine Learning devuelve instantáneamente la predicción exacta de kilogramos a asignar, garantizando una planificación 100% personalizada por orden.

---

## 8. Conclusiones y Recomendaciones

1. **Viabilidad Técnica:** Se demostró que es matemáticamente posible predecir la merma basándose en las variables del proceso, y que el tamaño de la orden (`peso_kg`) no es el único factor determinante, validando la necesidad de abandonar las tablas estáticas actuales.
2. **Limitación de Datos:** El MAE actual de prueba (11 kg) es elevado para un entorno de producción estricto. La causa raíz identificada es el tamaño del conjunto de datos histórico.
3. **Siguiente Paso (Recomendación Principal):** Antes de desplegar el modelo en el piso de producción, la planta debe establecer un protocolo estandarizado de recolección de datos durante los próximos meses. Al expandir el *dataset* de 200 a miles de registros, el algoritmo Random Forest actual incrementará drásticamente su precisión ($R^2$) y reducirá el margen de error, permitiendo una planificación de materiales altamente eficiente.